# vectorbt → canonical bundle → implementation-risk audit

This notebook runs one toy strategy in **vectorbt 1.1.0**, exports the result
with `backtest_auditor`, maps it to canonical bundle 1.0, checks it against an
independent pandas reference, and runs the implementation-risk audit on the
correct run and on a copy with an injected one-bar look-ahead.

Strategy and reference: [`toy_strategy.py`](toy_strategy.py) (SMA 20/50 cross,
a scheduled short, one-bar execution lag, USD 1.00 per fill). Shared steps:
[`audit_kit.py`](audit_kit.py). Outputs below come from
`jupyter nbconvert --execute --to notebook --inplace 01_vectorbt.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

import audit_kit as kit
import backtest_auditor
import toy_strategy as ts
import vectorbt as vbt
from backtest_auditor.implementation_risk import (
    ComparisonLevel,
    build_v1_5d2_service,
    export_vectorbt_portfolio,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
REPO = Path.cwd().resolve().parents[1]
package_dir = Path(backtest_auditor.__file__).resolve().parent
print(
    "backtest_auditor",
    backtest_auditor.__version__,
    "from",
    package_dir.relative_to(REPO),
)
print("vectorbt", vbt.__version__)

backtest_auditor 1.6.0 from src/backtest_auditor
vectorbt 1.1.0


## 1. The data and the independent reference

`toy_strategy.py` holds the bars and computes the expected result with pandas
and `decimal` only (no engine, no auditor): ten fills, five round trips
including one short, final equity USD 99,844.00.

In [2]:
prices = ts.load_prices()
expected = ts.expected_run()
print(len(prices), "bars,", prices.index[0], "to", prices.index[-1])
prices.head(3)

400 bars, 2024-01-02 21:00:00+00:00 to 2025-07-14 20:00:00+00:00


,open,high,low,close,volume
timestamp,,,,,
2024-01-02 21:00:00+00:00,100.00,100.23,99.79,100.00,1000000.0
2024-01-03 21:00:00+00:00,100.19,101.07,100.06,100.99,1000000.0
2024-01-04 21:00:00+00:00,100.69,101.40,97.34,97.43,1000000.0


## 2. The strategy in vectorbt

vectorbt computes both moving averages (`vbt.MA`) and simulates target
positions with `Portfolio.from_orders`: market fills at the bar close, a fixed
fee per fill.

In [3]:
def run_vectorbt(lag: int) -> vbt.Portfolio:
    """The toy strategy the vectorbt way; lag=0 is the look-ahead bug."""
    close = prices["close"]
    fast = vbt.MA.run(close, window=ts.FAST).ma.to_numpy()
    slow = vbt.MA.run(close, window=ts.SLOW).ma.to_numpy()
    long_signal = pd.Series((fast > slow).astype(int), index=close.index)
    sessions = (
        close.index.tz_convert(ts.EXCHANGE_TIMEZONE).normalize().tz_localize(None)
    )
    signal = long_signal.where(~ts.in_short_window(sessions), -1)
    target = signal.shift(lag).fillna(0) * ts.QUANTITY  # shares held at each close
    return vbt.Portfolio.from_orders(
        close,
        size=target,
        size_type="targetamount",
        fixed_fees=float(ts.FEE),
        init_cash=float(ts.INITIAL_CASH),
        freq="1D",
    )


portfolio = run_vectorbt(lag=1)
print("final value:", portfolio.final_value())

final value: 99844.0


## 3. Export, map to canonical bundle 1.0, check against the reference

`export_vectorbt_portfolio` reads the executed portfolio (it never re-runs it)
and refuses any vectorbt other than 1.1.0. The D2 adapter maps the export to
canonical artifacts; anything vectorbt does not record is marked unavailable,
and every reconstruction is reported as a warning.

In [4]:
EXPORTS = kit.OUTPUT / "vectorbt"
exported = export_vectorbt_portfolio(
    portfolio,
    EXPORTS / "export-correct",
    run_id="vectorbt-correct",
    instrument_id=ts.INSTRUMENT,
    base_currency=ts.CURRENCY,
    overwrite=True,
)
adapted = build_v1_5d2_service().adapt(exported)
print("adapter warnings:", [warning.code for warning in adapted.warnings])
bundle = kit.write_bundle(adapted.artifacts, EXPORTS / "vectorbt-correct.zip")
kit.bundle_summary(EXPORTS / "vectorbt-correct.zip")

adapter warnings: ['vectorbt.filled_orders_reconstructed', 'vectorbt.initial_return_omitted']


,file,records,sha256
0,cash_balances.csv,400,668dd0f982d4…
1,costs.csv,10,986c3f7cb4fc…
2,equity.csv,400,6ef8366737ce…
3,fills.csv,10,f6f76baf83df…
4,orders.csv,10,5c534e32bd97…
5,positions.csv,400,0de24adf461c…
6,returns.csv,399,4df3a1e546f2…
7,trade_fills.csv,0,b65094a5fdf1…
8,trades.csv,5,ffb841196995…


In [5]:
kit.compare_fills(bundle, expected)

,timestamp (UTC),engine,reference,same
0,2024-03-25 20:00,buy 100 @ 89.51,buy 100 @ 89.51,True
1,2024-06-19 20:00,sell 100 @ 97.33,sell 100 @ 97.33,True
2,2024-08-12 20:00,buy 100 @ 101.88,buy 100 @ 101.88,True
3,2024-09-25 20:00,sell 100 @ 94.88,sell 100 @ 94.88,True
4,2024-11-11 21:00,buy 100 @ 97.56,buy 100 @ 97.56,True
5,2025-01-24 21:00,sell 100 @ 94.96,sell 100 @ 94.96,True
6,2025-02-07 21:00,sell 100 @ 96.59,sell 100 @ 96.59,True
7,2025-03-07 21:00,buy 100 @ 91.56,buy 100 @ 91.56,True
8,2025-04-15 20:00,buy 100 @ 102.53,buy 100 @ 102.53,True
9,2025-06-03 20:00,sell 100 @ 97.82,sell 100 @ 97.82,True


In [6]:
kit.final_value_check(bundle, expected)

,engine,reference
,,
final equity,99844,99844.00


## 4. Implementation-risk audit of the correct run

The reference bundle is built from `toy_strategy.py` alone. Levels are aligned
on economics (time, instrument, side), not on engine-local ids. Materiality
thresholds sit far above float noise and far below the effect of a one-bar
shift (see `audit_kit.MEASURED_FIELDS`). Levels compared for vectorbt:
fill, trade, position, cash, cost, equity, return.

How to read the summary: a score near zero means every measured difference
is float noise below its threshold. No score with uncertainty `not_assessed`
means there was no numeric observation to classify, so read it with the
findings: with no `adverse` or `evidence_gap` finding it is exact agreement on
every measured field; with `evidence_gap` findings it means records could not
be paired at all (the NautilusTrader look-ahead run shows that case).

In [7]:
LEVELS = (
    ComparisonLevel.FILL,
    ComparisonLevel.TRADE,
    ComparisonLevel.POSITION,
    ComparisonLevel.CASH,
    ComparisonLevel.COST,
    ComparisonLevel.EQUITY,
    ComparisonLevel.RETURN,
)
reference_bundle = kit.reference_artifacts()
correct = kit.implementation_risk(
    reference_bundle, bundle, LEVELS, name="vectorbt-correct"
)
kit.summary(correct)

,implementation_risk_score,uncertainty,score_withheld_reason,supporting findings
candidate,,,,
vectorbt-correct,0.00000000077460559,low,None,206


In [8]:
kit.levels_table(correct)

,status,matched_equal,matched_different
level,,,
fill,compared,10,0
trade,compared,0,5
position,compared,0,400
cash,compared,0,400
cost,compared,10,0
equity,compared,0,400
return,compared,194,205


`matched_different` counts matched records in which any field differs,
including fields that only one side fills in (the reference leaves market
values and exposures empty) and labels that differ by design (a trade's
grouping method). Which fields differ, and whether the measurement policy
measures them:

In [9]:
kit.differing_fields(correct)

,level,field,records,measured
0,trade,grouping_method,5,False
1,position,market_price,400,False
2,position,market_value,400,False
3,cash,settled,400,False
4,equity,gross_exposure,400,False
5,equity,net_exposure,400,False
6,return,value,205,True


In [10]:
print(kit.report_excerpt(correct))

# Backtest Implementation-Risk Report

- Report schema: `1.0`
- Reconciliation: `toy-ma-cross`
- Reference: `reference-pandas` (pandas-reference)
- Candidates: 1
- Overall status: not produced

> The report identifies implementation divergence against a declared reference baseline. It does not establish that any engine is correct, prove causation, certify strategy skill, or predict future performance.

## Policy identity

| Policy | SHA-256 |
|---|---|
| Normalization | `dfe9df9b13f956106e66a34a3576a925a428a142ac3f52751fb797473acfd75a` |
| Comparison | `ef1ba80428ca34e377643a635ae0ab4a5523834289ee75b8e4e6a31fdaf9ec8d` |
| Measurement | `641a183ee09ae612f72ac504c565adbc6df9cc7dd9a5df8742d3859a6b684e96` |
| Assessment | `e4b5e2e8b0c99b233f5a0324d3511321f91c105b92b8b5c7d557683b9c3f8556` |

## Candidate `vectorbt-correct`

- Engine: vectorbt
- Implementation-risk score: 0.00000000077460559
- Uncertainty: low

### Comparison levels

| Level | Status | Alignment | Exact match | Records |
|--

## 5. The same strategy with a one-bar look-ahead

The defect: the position follows the signal of the same bar, so the strategy
trades on a close it could not have known. Every fill moves one bar earlier.

In [11]:
exported = export_vectorbt_portfolio(
    run_vectorbt(lag=0),
    EXPORTS / "export-lookahead",
    run_id="vectorbt-lookahead",
    instrument_id=ts.INSTRUMENT,
    base_currency=ts.CURRENCY,
    overwrite=True,
)
lookahead_bundle = kit.write_bundle(
    build_v1_5d2_service().adapt(exported).artifacts, EXPORTS / "vectorbt-lookahead.zip"
)
lookahead = kit.implementation_risk(
    reference_bundle, lookahead_bundle, LEVELS, name="vectorbt-lookahead"
)
kit.summary(lookahead)

,implementation_risk_score,uncertainty,score_withheld_reason,adverse findings,evidence_gap findings
candidate,,,,,
vectorbt-lookahead,None,high,score withheld because selected divergence evidence is incomplete,907,5


In [12]:
kit.families_table(lookahead)

,status,maximum threshold ratio,score contribution
family,,,
transaction_cost,partial,NaN,NaN
sizing,partial,200,25
accounting,partial,1041300,25
return,measured,224327.5181765377451078575,25


In [13]:
print(kit.report_findings(lookahead, 4))

## Independent findings

| Code | Kind | Candidate | Family | Summary | Limitation | Action |
|---|---|---|---|---|---|---|
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | vectorbt-lookahead | transaction_cost | transaction_cost evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_FAMILY_EVIDENCE_IMPAIRED | evidence_gap | vectorbt-lookahead | sizing | sizing evidence is partial. | Materiality and scoring are bounded by incomplete evidence. | Resolve unavailable, one-sided, ambiguous, or non-numeric evidence. |
| IMPLEMENTATION_DIVERGENCE_EXTREME | adverse | vectorbt-lookahead | sizing | sizing quantity divergence is extreme at 200× threshold. | The classification is conditional on the supplied field-specific threshold. | Review the canonical records and the declared materiality threshold. |
| IMPLEMENTATION_DIVERGENCE_EXTREME | adverse | vectorbt-lookahead |

## 6. The same audit from the command line

The CLI reads bundle ZIPs and the four policies as JSON files. The package
serializes its own policy objects (`normalization_plan_to_json` and its three
siblings), so the policies above can be written out and the correct run
audited again by `backtest-auditor implementation-risk`:

In [14]:
import shutil

CLI_DIR = EXPORTS / "cli"
shutil.rmtree(CLI_DIR, ignore_errors=True)  # list only what this run writes
kit.write_bundle(kit.restrict(reference_bundle, LEVELS), CLI_DIR / "reference.zip")
kit.write_bundle(kit.restrict(bundle, LEVELS), CLI_DIR / "candidate.zip")
kit.policies(reference_bundle, bundle, LEVELS).write(CLI_DIR)
print(sorted(path.name for path in CLI_DIR.iterdir()))
print((CLI_DIR / "comparison-policy.json").read_text()[:240], "...")

['assessment-policy.json', 'candidate.zip', 'comparison-policy.json', 'measurement-policy.json', 'normalization-plan.json', 'reference.zip']
{"comparison_schema_version":"1.0","levels":[{"alignment_mode":"semantic","level":"fill","rationale":"Engine-local identifiers differ by design; align on economics."},{"alignment_mode":"semantic","level":"trade","rationale":"Engine-local id ...


In [15]:
import json
import subprocess
import sys

command = [
    str(Path(sys.executable).with_name("backtest-auditor")),
    "implementation-risk",
    "reference.zip",
    "candidate.zip",
    "--normalization-plan", "normalization-plan.json",
    "--comparison-policy", "comparison-policy.json",
    "--measurement-policy", "measurement-policy.json",
    "--assessment-policy", "assessment-policy.json",
    "--format", "json",
    "--output", "report.json",
    "--force",
]  # fmt: skip
completed = subprocess.run(command, cwd=CLI_DIR, capture_output=True, text=True)
print("exit code:", completed.returncode, completed.stdout, completed.stderr)
cli_report = json.loads((CLI_DIR / "report.json").read_text(encoding="utf-8"))
kit.validate_report(cli_report)
print(
    "CLI report equals the report of the Python call above:",
    cli_report == correct.report,
)

exit code: 0  
CLI report equals the report of the Python call above: True


## What this shows, and what it does not

- The vectorbt run and the independent reference agree on every measured
  field at every level compared above (other differences are confined to
  fields the reference leaves empty, labels that differ by design, or float
  noise below the thresholds), and the audit names no defect.
- The injected one-bar look-ahead produces adverse findings in the sizing, accounting and return
  families, and the score is withheld because evidence on some
  levels became one-sided.
- The audit compares evidence against a declared reference under explicit
  thresholds. It does not prove either implementation correct, and it says
  nothing about whether the strategy is any good.
- The full JSON and Markdown reports of both runs are in `_output/` after
  executing this notebook.